# Data Sanity Pass (Issue #6)

As a modeller, I want to know the shape, target rate, and missing-value encoding before I train anything.

This is **not** the full EDA. This is a sanity pass on `train.csv`, done *before* splitting the data, so we understand what we're working with. It answers exactly five questions:

1. How many rows and columns? Is there a column that uniquely identifies a row?
2. What fraction of rows are positive (`target = 1`)?
3. How are missing values represented? Check for blanks, `NaN`, and sentinel values, counted per column.
4. What kinds of column are there, and how do we tell them apart? Specifically: which are categories (codes, like postcodes) vs. quantities (where bigger means more)?

These are structural facts about the file (shapes, dtypes, missingness, target rate), so answering them on the whole of `train.csv` is fine and does not compromise the test set.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/train.csv")

n_rows, n_cols = df.shape
has_unique_id = "id" in df.columns and df["id"].is_unique
print(f"Rows: {n_rows}, Columns: {n_cols}")
print(f"'id' uniquely identifies each row: {has_unique_id}")

positive_rate = df["target"].mean()
print(f"Positive rate (target=1): {positive_rate:.4%}")

feature_cols = [c for c in df.columns if c not in ("id", "target")]
blank_counts = (df[feature_cols] == "").sum()
nan_counts = df[feature_cols].isna().sum()
sentinel_counts = (df[feature_cols] == -1).sum()
missing_summary = pd.DataFrame({"blank": blank_counts, "nan": nan_counts, "sentinel_-1": sentinel_counts})
missing_summary = missing_summary[missing_summary.sum(axis=1) > 0]
print(missing_summary)

categorical_cols = [c for c in feature_cols if c.endswith("_cat") or c.endswith("_bin")]
quantity_cols = [c for c in feature_cols if c not in categorical_cols]
print("Categorical/label columns:", categorical_cols)
print("Quantity columns:", quantity_cols)

Rows: 595212, Columns: 59
'id' uniquely identifies each row: True
Positive rate (target=1): 3.6448%
               blank  nan  sentinel_-1
ps_ind_02_cat      0    0          216
ps_ind_04_cat      0    0           83
ps_ind_05_cat      0    0         5809
ps_reg_03          0    0       107772
ps_car_01_cat      0    0          107
ps_car_02_cat      0    0            5
ps_car_03_cat      0    0       411231
ps_car_05_cat      0    0       266551
ps_car_07_cat      0    0        11489
ps_car_09_cat      0    0          569
ps_car_11          0    0            5
ps_car_12          0    0            1
ps_car_14          0    0        42620
Categorical/label columns: ['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_ind_06_bin', 'ps_ind_07_bin', 'ps_ind_08_bin', 'ps_ind_09_bin', 'ps_ind_10_bin', 'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_16_bin', 'ps_ind_17_bin', 'ps_ind_18_bin', 'ps_car_01_cat', 'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_04_cat', 'ps_car_05_cat', 'ps